# RTE ASR uncertainty workflow

This notebook runs the absolute sustainability ratio (ASR) uncertainty analysis for the RTE publication case study. It assumes the external LCA Monte Carlo sources have already been generated by `rte_lca_monte_carlo_publication.ipynb` in the same workspace.


## `pyaesa` installation

This notebook uses the `pyaesa` Python package. Install the release from PyPI before running the workflow:

```bash
python -m pip install pyaesa
```

For package documentation, API reference, and tutorials, see [pyaesa.readthedocs.io](https://pyaesa.readthedocs.io/). 

The source code is available on GitHub at [AESAtoolkit/pyaesa](https://github.com/AESAtoolkit/pyaesa).

## Notebook configuration

In [ ]:
from pyaesa import (
    set_workspace,
    download_ar6,
    download_mrio,
    download_pop_gdp,
    process_mrio,
    process_pop_gdp,
    deterministic_asocc,
    disaggregate_asocc,
    uncertainty_asr,
)

WORKSPACE_TOP = r"C:\Users\Erwan\Documents\UNCASExt_demo"  # replace with your workspace path
YEARS_ASOCC = range(1995, 2061)
YEARS_ASR = range(2019, 2061)
R_C = ["FR"]
S_P = ["Electricity"]
LCIA_METHODS = ["pb_lcia", "gwp100_lcia"]

## Initialize the workspace

`set_workspace(...)` creates or selects the pyaesa workspace where the LCA notebook staged the external LCA sources and where the ASR outputs will be written.


In [ ]:
set_workspace(WORKSPACE_TOP)


## Download and process `pyaesa` input data


### AR6 climate scenario data

Download the AR6 climate scenario data required for dynamic AR6 carrying capacities.


In [ ]:
download_ar6()

### Population and GDP data

Download and process population and GDP data needed for computing allocated shares.


In [ ]:
download_pop_gdp()

In [ ]:
process_pop_gdp()

### MRIO data

Download and process the MRIO tables needed for computing allocated shares.

The `elec` EXIOBASE sector aggregation groups the different EXIOBASE electricity sectors together (aggregation csv provided by `pyaesa`).

The `oecd_d` EXIOBASE sector aggregation groups EXIOBASE electricity, gas, and water sectors to match OECD ICIO sector D, `Electricity, gas, steam and air conditioning supply` (aggregation csv provided by `pyaesa`).

The OECD `fr` region aggregation renames the OECD ICIO region code `FRA` to `FR` so it matches EXIOBASE France code in the disaggregation step.


In [ ]:
download_mrio("exiobase_3102_ixi")

In [ ]:
download_mrio("oecd_v2025")

In [ ]:
process_mrio(
    "exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    lcia_method=LCIA_METHODS,
)


In [ ]:
process_mrio(
    "exiobase_3102_ixi",
    agg_sec=True,
    agg_version="oecd_d",
)


In [ ]:
process_mrio(
    "oecd_v2025",
    agg_reg=True,
    agg_version="fr",
)


## Prepare disaggregated aSoCC for inter-MRIO uncertainty

For each SSP, the notebook computes three deterministic aSoCC prerequisite scopes: OECD ICIO sector D target, EXIOBASE grouped to OECD ICIO sector D, and EXIOBASE electricity reference scope.

`disaggregate_asocc(...)` then uses the two EXIOBASE reference scopes to split the OECD ICIO sector D aSoCC target to EXIOBASE electricity sector resolution and writes the `oecd_electricity` source used by inter-MRIO uncertainty.


### SSP2

Run the deterministic prerequisite scopes, then create the disaggregated OECD ICIO electricity source for later use by inter-MRIO uncertainty.


#### OECD ICIO sector D aSoCC target


In [ ]:
deterministic_asocc(
    project_name="rte_scenarios_ssp2",
    source="oecd_v2025",
    agg_reg=True,
    agg_version="fr",
    years=YEARS_ASOCC,
    fu_code="L2.c.b",
    s_p=["D"],
    r_c=R_C,
    ssp_scenario=["SSP2"],
    figures=False,
)


#### EXIOBASE grouped to OECD ICIO sector D


In [ ]:
deterministic_asocc(
    project_name="rte_scenarios_ssp2",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="oecd_d",
    years=YEARS_ASOCC,
    fu_code="L2.c.b",
    s_p=["D"],
    r_c=R_C,
    ssp_scenario=["SSP2"],
    figures=False,
)


#### EXIOBASE electricity reference


In [ ]:
deterministic_asocc(
    project_name="rte_scenarios_ssp2",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASOCC,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method=LCIA_METHODS,
    ssp_scenario=["SSP2"],
    figures=False,
)


#### Disaggregate OECD ICIO sector D to EXIOBASE electricity


In [ ]:
disaggregate_asocc(
    disaggregation_config={
        "target_agg_run": {
            "source": "oecd_v2025",
            "agg_reg": True,
            "agg_version": "fr",
            "s_p": ["D"],
        },
        "ref_agg_run": {
            "source": "exiobase_3102_ixi",
            "agg_sec": True,
            "agg_version": "oecd_d",
            "s_p": ["D"],
        },
        "ref_disagg_run": {
            "source": "exiobase_3102_ixi",
            "agg_sec": True,
            "agg_version": "elec",
            "s_p": S_P,
        },
        "disaggregation_specs": [{"agg_sector_label": "D", "disagg_sector_label": "Electricity"}],
        "new_disagg_version_name": "oecd_electricity",
    },
    base_asocc_args={
        "project_name": "rte_scenarios_ssp2",
        "years": YEARS_ASOCC,
        "fu_code": "L2.c.b",
        "r_c": R_C,
        "ssp_scenario": ["SSP2"],
    },
    figures=False,
)


### SSP1

Run the deterministic prerequisite scopes, then create the disaggregated OECD ICIO electricity source for later use by inter-MRIO uncertainty.


#### OECD ICIO sector D aSoCC target


In [ ]:
deterministic_asocc(
    project_name="rte_scenarios_ssp1",
    source="oecd_v2025",
    agg_reg=True,
    agg_version="fr",
    years=YEARS_ASOCC,
    fu_code="L2.c.b",
    s_p=["D"],
    r_c=R_C,
    ssp_scenario=["SSP1"],
    figures=False,
)


#### EXIOBASE grouped to OECD ICIO sector D


In [ ]:
deterministic_asocc(
    project_name="rte_scenarios_ssp1",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="oecd_d",
    years=YEARS_ASOCC,
    fu_code="L2.c.b",
    s_p=["D"],
    r_c=R_C,
    ssp_scenario=["SSP1"],
    figures=False,
)


#### EXIOBASE electricity reference


In [ ]:
deterministic_asocc(
    project_name="rte_scenarios_ssp1",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASOCC,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method=LCIA_METHODS,
    ssp_scenario=["SSP1"],
    figures=False,
)


#### Disaggregate OECD ICIO sector D to EXIOBASE electricity


In [ ]:
disaggregate_asocc(
    disaggregation_config={
        "target_agg_run": {
            "source": "oecd_v2025",
            "agg_reg": True,
            "agg_version": "fr",
            "s_p": ["D"],
        },
        "ref_agg_run": {
            "source": "exiobase_3102_ixi",
            "agg_sec": True,
            "agg_version": "oecd_d",
            "s_p": ["D"],
        },
        "ref_disagg_run": {
            "source": "exiobase_3102_ixi",
            "agg_sec": True,
            "agg_version": "elec",
            "s_p": S_P,
        },
        "disaggregation_specs": [{"agg_sector_label": "D", "disagg_sector_label": "Electricity"}],
        "new_disagg_version_name": "oecd_electricity",
    },
    base_asocc_args={
        "project_name": "rte_scenarios_ssp1",
        "years": YEARS_ASOCC,
        "fu_code": "L2.c.b",
        "r_c": R_C,
        "ssp_scenario": ["SSP1"],
    },
    figures=False,
)


### SSP5

Run the deterministic prerequisite scopes, then create the disaggregated OECD ICIO electricity source for later use by inter-MRIO uncertainty.


#### OECD ICIO sector D aSoCC target


In [ ]:
deterministic_asocc(
    project_name="rte_scenarios_ssp5",
    source="oecd_v2025",
    agg_reg=True,
    agg_version="fr",
    years=YEARS_ASOCC,
    fu_code="L2.c.b",
    s_p=["D"],
    r_c=R_C,
    ssp_scenario=["SSP5"],
    figures=False,
)


#### EXIOBASE grouped to OECD ICIO sector D


In [ ]:
deterministic_asocc(
    project_name="rte_scenarios_ssp5",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="oecd_d",
    years=YEARS_ASOCC,
    fu_code="L2.c.b",
    s_p=["D"],
    r_c=R_C,
    ssp_scenario=["SSP5"],
    figures=False,
)


#### EXIOBASE electricity reference


In [ ]:
deterministic_asocc(
    project_name="rte_scenarios_ssp5",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASOCC,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method=LCIA_METHODS,
    ssp_scenario=["SSP5"],
    figures=False,
)


#### Disaggregate OECD ICIO sector D to EXIOBASE electricity


In [ ]:
disaggregate_asocc(
    disaggregation_config={
        "target_agg_run": {
            "source": "oecd_v2025",
            "agg_reg": True,
            "agg_version": "fr",
            "s_p": ["D"],
        },
        "ref_agg_run": {
            "source": "exiobase_3102_ixi",
            "agg_sec": True,
            "agg_version": "oecd_d",
            "s_p": ["D"],
        },
        "ref_disagg_run": {
            "source": "exiobase_3102_ixi",
            "agg_sec": True,
            "agg_version": "elec",
            "s_p": S_P,
        },
        "disaggregation_specs": [{"agg_sector_label": "D", "disagg_sector_label": "Electricity"}],
        "new_disagg_version_name": "oecd_electricity",
    },
    base_asocc_args={
        "project_name": "rte_scenarios_ssp5",
        "years": YEARS_ASOCC,
        "fu_code": "L2.c.b",
        "r_c": R_C,
        "ssp_scenario": ["SSP5"],
    },
    figures=False,
)


## Run ASR uncertainty scenarios

Each cell below runs one publication case. For each RTE version and SSP, the order is `pb_lcia` with static carrying capacities, `gwp100_lcia` with dynamic AR6 carrying capacities, then `gwp100_lcia` with static carrying capacities. Sobol is activated for all cases, while Monte Carlo convergence settings use pyaesa defaults.


### REF_M0


#### SSP2


##### `pb_lcia`, static carrying capacities


In [ ]:
print("Running rte_scenarios_ssp2 | rte_reference_m0_ssp2 | pb_lcia | static CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp2",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="pb_lcia",
    base_asocc_args={"ssp_scenario": ["SSP2"]},
    lca_args={"external_lca": {"version_name": "rte_reference_m0_ssp2"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
    },
    sobol_parameters={"active": True},
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


##### `gwp100_lcia`, dynamic AR6 carrying capacities


In [ ]:
print("Running rte_scenarios_ssp2 | rte_reference_m0_ssp2 | gwp100_lcia | dynamic AR6 CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp2",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="gwp100_lcia",
    base_asocc_args={"ssp_scenario": ["SSP2"]},
    base_cc_args={
        "static": {"active": False},
        "dynamic_ar6": {"active": True, "ssp_scenario": ["SSP2"]},
    },
    lca_args={"external_lca": {"version_name": "rte_reference_m0_ssp2"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
        "ar6_cc_uncertainty_sources": {
            "dynamic_ar6_cc_uncertainty": {"category_uncertainty": True},
        },
    },
    sobol_parameters={"active": True},
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


##### `gwp100_lcia`, static carrying capacities


In [ ]:
print("Running rte_scenarios_ssp2 | rte_reference_m0_ssp2 | gwp100_lcia | static CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp2",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="gwp100_lcia",
    base_asocc_args={"ssp_scenario": ["SSP2"]},
    lca_args={"external_lca": {"version_name": "rte_reference_m0_ssp2"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
    },
    sobol_parameters={"active": True},
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


#### SSP1


##### `pb_lcia`, static carrying capacities


In [ ]:
print("Running rte_scenarios_ssp1 | rte_reference_m0_ssp1 | pb_lcia | static CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp1",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="pb_lcia",
    base_asocc_args={"ssp_scenario": ["SSP1"]},
    lca_args={"external_lca": {"version_name": "rte_reference_m0_ssp1"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
    },
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


##### `gwp100_lcia`, dynamic AR6 carrying capacities


In [ ]:
print("Running rte_scenarios_ssp1 | rte_reference_m0_ssp1 | gwp100_lcia | dynamic AR6 CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp1",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="gwp100_lcia",
    base_asocc_args={"ssp_scenario": ["SSP1"]},
    base_cc_args={
        "static": {"active": False},
        "dynamic_ar6": {"active": True, "ssp_scenario": ["SSP1"]},
    },
    lca_args={"external_lca": {"version_name": "rte_reference_m0_ssp1"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
        "ar6_cc_uncertainty_sources": {
            "dynamic_ar6_cc_uncertainty": {"category_uncertainty": True},
        },
    },
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


##### `gwp100_lcia`, static carrying capacities


In [ ]:
print("Running rte_scenarios_ssp1 | rte_reference_m0_ssp1 | gwp100_lcia | static CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp1",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="gwp100_lcia",
    base_asocc_args={"ssp_scenario": ["SSP1"]},
    lca_args={"external_lca": {"version_name": "rte_reference_m0_ssp1"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
    },
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


#### SSP5


##### `pb_lcia`, static carrying capacities


In [ ]:
print("Running rte_scenarios_ssp5 | rte_reference_m0_ssp5 | pb_lcia | static CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp5",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="pb_lcia",
    base_asocc_args={"ssp_scenario": ["SSP5"]},
    lca_args={"external_lca": {"version_name": "rte_reference_m0_ssp5"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
    },
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


##### `gwp100_lcia`, dynamic AR6 carrying capacities


In [ ]:
print("Running rte_scenarios_ssp5 | rte_reference_m0_ssp5 | gwp100_lcia | dynamic AR6 CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp5",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="gwp100_lcia",
    base_asocc_args={"ssp_scenario": ["SSP5"]},
    base_cc_args={
        "static": {"active": False},
        "dynamic_ar6": {"active": True, "ssp_scenario": ["SSP5"]},
    },
    lca_args={"external_lca": {"version_name": "rte_reference_m0_ssp5"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
        "ar6_cc_uncertainty_sources": {
            "dynamic_ar6_cc_uncertainty": {"category_uncertainty": True},
        },
    },
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


##### `gwp100_lcia`, static carrying capacities


In [ ]:
print("Running rte_scenarios_ssp5 | rte_reference_m0_ssp5 | gwp100_lcia | static CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp5",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="gwp100_lcia",
    base_asocc_args={"ssp_scenario": ["SSP5"]},
    lca_args={"external_lca": {"version_name": "rte_reference_m0_ssp5"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
    },
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


### REF_N03


#### SSP2


##### `pb_lcia`, static carrying capacities


In [ ]:
print("Running rte_scenarios_ssp2 | rte_reference_n03_ssp2 | pb_lcia | static CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp2",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="pb_lcia",
    base_asocc_args={"ssp_scenario": ["SSP2"]},
    lca_args={"external_lca": {"version_name": "rte_reference_n03_ssp2"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
    },
    sobol_parameters={"active": True},
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


##### `gwp100_lcia`, dynamic AR6 carrying capacities


In [ ]:
print("Running rte_scenarios_ssp2 | rte_reference_n03_ssp2 | gwp100_lcia | dynamic AR6 CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp2",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="gwp100_lcia",
    base_asocc_args={"ssp_scenario": ["SSP2"]},
    base_cc_args={
        "static": {"active": False},
        "dynamic_ar6": {"active": True, "ssp_scenario": ["SSP2"]},
    },
    lca_args={"external_lca": {"version_name": "rte_reference_n03_ssp2"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
        "ar6_cc_uncertainty_sources": {
            "dynamic_ar6_cc_uncertainty": {"category_uncertainty": True},
        },
    },
    sobol_parameters={"active": True},
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


##### `gwp100_lcia`, static carrying capacities


In [ ]:
print("Running rte_scenarios_ssp2 | rte_reference_n03_ssp2 | gwp100_lcia | static CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp2",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="gwp100_lcia",
    base_asocc_args={"ssp_scenario": ["SSP2"]},
    lca_args={"external_lca": {"version_name": "rte_reference_n03_ssp2"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
    },
    sobol_parameters={"active": True},
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


#### SSP1


##### `pb_lcia`, static carrying capacities


In [ ]:
print("Running rte_scenarios_ssp1 | rte_reference_n03_ssp1 | pb_lcia | static CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp1",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="pb_lcia",
    base_asocc_args={"ssp_scenario": ["SSP1"]},
    lca_args={"external_lca": {"version_name": "rte_reference_n03_ssp1"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
    },
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


##### `gwp100_lcia`, dynamic AR6 carrying capacities


In [ ]:
print("Running rte_scenarios_ssp1 | rte_reference_n03_ssp1 | gwp100_lcia | dynamic AR6 CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp1",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="gwp100_lcia",
    base_asocc_args={"ssp_scenario": ["SSP1"]},
    base_cc_args={
        "static": {"active": False},
        "dynamic_ar6": {"active": True, "ssp_scenario": ["SSP1"]},
    },
    lca_args={"external_lca": {"version_name": "rte_reference_n03_ssp1"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
        "ar6_cc_uncertainty_sources": {
            "dynamic_ar6_cc_uncertainty": {"category_uncertainty": True},
        },
    },
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


##### `gwp100_lcia`, static carrying capacities


In [ ]:
print("Running rte_scenarios_ssp1 | rte_reference_n03_ssp1 | gwp100_lcia | static CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp1",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="gwp100_lcia",
    base_asocc_args={"ssp_scenario": ["SSP1"]},
    lca_args={"external_lca": {"version_name": "rte_reference_n03_ssp1"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
    },
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


#### SSP5


##### `pb_lcia`, static carrying capacities


In [ ]:
print("Running rte_scenarios_ssp5 | rte_reference_n03_ssp5 | pb_lcia | static CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp5",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="pb_lcia",
    base_asocc_args={"ssp_scenario": ["SSP5"]},
    lca_args={"external_lca": {"version_name": "rte_reference_n03_ssp5"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
    },
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


##### `gwp100_lcia`, dynamic AR6 carrying capacities


In [ ]:
print("Running rte_scenarios_ssp5 | rte_reference_n03_ssp5 | gwp100_lcia | dynamic AR6 CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp5",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="gwp100_lcia",
    base_asocc_args={"ssp_scenario": ["SSP5"]},
    base_cc_args={
        "static": {"active": False},
        "dynamic_ar6": {"active": True, "ssp_scenario": ["SSP5"]},
    },
    lca_args={"external_lca": {"version_name": "rte_reference_n03_ssp5"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
        "ar6_cc_uncertainty_sources": {
            "dynamic_ar6_cc_uncertainty": {"category_uncertainty": True},
        },
    },
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)


##### `gwp100_lcia`, static carrying capacities


In [ ]:
print("Running rte_scenarios_ssp5 | rte_reference_n03_ssp5 | gwp100_lcia | static CC")
uncertainty_asr(
    project_name="rte_scenarios_ssp5",
    source="exiobase_3102_ixi",
    agg_sec=True,
    agg_version="elec",
    years=YEARS_ASR,
    fu_code="L2.c.b",
    s_p=S_P,
    r_c=R_C,
    lcia_method="gwp100_lcia",
    base_asocc_args={"ssp_scenario": ["SSP5"]},
    lca_args={"external_lca": {"version_name": "rte_reference_n03_ssp5"}},
    uncertainty_config={
        "mc_parameters": {"convergence": {"max_runs": 300_000}},
        "asocc_uncertainty_sources": {
            "lcia_uncertainty": {
                "active": True,
                "sector_cov_mapping": {"Electricity": "Electricity"},
            },
            "inter_mrio_uncertainty": {"active": True, "alternate_source": "oecd_electricity"},
        },
    },
    output_format="parquet",
    figure_format={"format": "svg", "dpi": 1000},
)
